## Graph Intelligence: Vercel AI SDK + Neo4j
Three patterns for integrating Neo4j with the [Vercel AI SDK](https://sdk.vercel.ai/docs):
1. **MCP Agent** — query the graph via the `neo4j-mcp-server` (HTTP + Basic Auth)
2. **Custom Tools** — define Cypher-backed tools alongside MCP tools in one agent
3. **Memory Agent** — persist and retrieve conversation context in Neo4j using `neo4j-driver`

See [`providers.mjs`](providers.mjs) to configure a different LLM provider (Google Gemini, Anthropic, Mistral).

### 1. Environment Setup
Installs the required packages.  
• **ai** — Vercel AI SDK  
• **@ai-sdk/openai** — OpenAI provider  
• **@ai-sdk/mcp** — MCP client integration  
• **neo4j-driver** — Neo4j connectivity and graph-native memory  
• **neo4j-mcp-server** — Python MCP server that exposes Neo4j as agent tools

In [ ]:
import os, subprocess, sys, shutil

# ── Node.js: ensure 20+ is on PATH ────────────────────────────────────────────
node = shutil.which('node')
if not node:
    node20_bin = os.path.expanduser('~/node20/bin')
    if os.path.isdir(node20_bin):
        os.environ['PATH'] = node20_bin + ':' + os.environ.get('PATH', '')
        node = shutil.which('node')

if node:
    v = subprocess.run(['node', '--version'], capture_output=True, text=True).stdout.strip()
    print(f'Node.js: {v}  ({node})')
else:
    print('ERROR: Node.js not found. Install Node.js 20+ from https://nodejs.org')

# ── npm packages ───────────────────────────────────────────────────────────────
pkg_dir = os.getcwd()
if not os.path.isdir(os.path.join(pkg_dir, 'node_modules', '@ai-sdk', 'mcp')):
    print('Installing npm packages...')
    r = subprocess.run(['npm', 'install'], cwd=pkg_dir, env=os.environ,
                       capture_output=True, text=True)
    print(r.stdout[-500:] if r.stdout else r.stderr[-500:])
else:
    print('npm packages already installed ✓')

# ── neo4j-mcp-server (Python) ─────────────────────────────────────────────────
r3 = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--ignore-requires-python', 'neo4j-mcp-server'],
    capture_output=True, text=True
)
print('neo4j-mcp-server:', 'installed ✓' if r3.returncode == 0 else r3.stderr[:200])

### 2. Configuration
- The **companies demo database** is read-only and public — no credentials needed beyond the defaults.
- **Memory** requires a separate writable Neo4j instance (configure below in the Memory section).
- Change `AI_PROVIDER` / `AI_MODEL` to switch LLMs.

In [ ]:
import os
from getpass import getpass

# ── Knowledge Graph (read-only, public companies demo) ────────────────────────
os.environ.setdefault('NEO4J_URI',      'neo4j+s://demo.neo4jlabs.com:7687')
os.environ.setdefault('NEO4J_USERNAME', 'companies')
os.environ.setdefault('NEO4J_PASSWORD', 'companies')

# ── LLM provider ─────────────────────────────────────────────────────────────
os.environ.setdefault('AI_PROVIDER', 'openai')
os.environ.setdefault('AI_MODEL',    'gpt-4o')

# Set your API key if not already in the environment
key_var = {'openai': 'OPENAI_API_KEY', 'google': 'GOOGLE_GENERATIVE_AI_API_KEY',
           'anthropic': 'ANTHROPIC_API_KEY', 'mistral': 'MISTRAL_API_KEY'}
provider = os.environ['AI_PROVIDER']
env_key  = key_var.get(provider, 'OPENAI_API_KEY')
if not os.environ.get(env_key):
    os.environ[env_key] = getpass(f'{env_key}: ')

print(f"Provider: {os.environ['AI_PROVIDER']} / {os.environ['AI_MODEL']}")
print(f"Neo4j:    {os.environ['NEO4J_URI']}")

### 3. MCP Server
> The `neo4j-mcp-server` runs as a background HTTP server. We kill any existing process on the port before starting to make the cell re-runnable.

In [ ]:
import subprocess, os, time, signal

MCP_PORT = int(os.environ.get('MCP_PORT', '8080'))

# Kill any process already using the port
kill = subprocess.run(['lsof', '-ti', f'tcp:{MCP_PORT}'], capture_output=True, text=True)
for pid in kill.stdout.split():
    try:
        os.kill(int(pid), signal.SIGTERM)
    except ProcessLookupError:
        pass

os.environ['MCP_PORT'] = str(MCP_PORT)

proc = subprocess.Popen(
    ['python', '-m', 'neo4j_mcp',
     '--uri',      os.environ['NEO4J_URI'],
     '--username', os.environ['NEO4J_USERNAME'],
     '--password', os.environ['NEO4J_PASSWORD'],
     '--port',     str(MCP_PORT),
     '--transport', 'http'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
time.sleep(2)
print(f'neo4j-mcp-server PID {proc.pid} listening on :{MCP_PORT} ✓')

### 4. MCP Agent
[`1-mcp-agent.mjs`](1-mcp-agent.mjs) connects to the MCP server via HTTP Basic Auth, asks the agent how many organisations are in the database, and prints the answer.

In [ ]:
import subprocess, os

r = subprocess.run(
    ['node', '1-mcp-agent.mjs'],
    cwd=os.getcwd(), env=os.environ,
    capture_output=True, text=True, timeout=120,
)
print(r.stdout)
if r.stderr:
    print('STDERR:', r.stderr[:500])

### 5. Custom Tools
Beyond MCP, you can define Cypher-backed tools with `tool()` + `inputSchema: jsonSchema({...})` and mix them with MCP tools in the same agent — no Zod required.  
[`2-custom-tools-agent.mjs`](2-custom-tools-agent.mjs) adds a `getInvestments` tool that queries investment relationships directly via `neo4j-driver`.

In [ ]:
import subprocess, os

r = subprocess.run(
    ['node', '2-custom-tools-agent.mjs'],
    cwd=os.getcwd(), env=os.environ,
    capture_output=True, text=True, timeout=120,
)
print(r.stdout)
if r.stderr:
    print('STDERR:', r.stderr[:500])

### 6. Persistent Memory Agent
The Vercel AI SDK has no built-in plugin lifecycle, but we can implement a **before/after hook** pattern around each `generateText` call.  
Memory is stored directly in Neo4j using `neo4j-driver` — no extra npm packages needed:  
`(:MemorySession {id})-[:HAS_MESSAGE]->(:MemoryMessage {role, content, timestamp})`

[`3-memory-agent.mjs`](3-memory-agent.mjs) demonstrates a two-turn conversation where Turn 2 correctly references context established in Turn 1.

In [ ]:
import os
from getpass import getpass

# Memory requires a writable Neo4j instance (separate from the read-only demo DB).
if not os.environ.get('MEMORY_NEO4J_URI'):
    os.environ['MEMORY_NEO4J_URI']      = input('Memory Neo4j URI (neo4j+s://...): ').strip()
    os.environ['MEMORY_NEO4J_USERNAME'] = input('Memory Neo4j username [neo4j]: ').strip() or 'neo4j'
    os.environ['MEMORY_NEO4J_PASSWORD'] = getpass('Memory Neo4j password: ')
    os.environ['MEMORY_NEO4J_DATABASE'] = input('Memory database [neo4j]: ').strip() or 'neo4j'

print(f"Memory URI: {os.environ.get('MEMORY_NEO4J_URI', 'not set')}")
print(f"Memory DB:  {os.environ.get('MEMORY_NEO4J_DATABASE', 'not set')}")

In [ ]:
import subprocess, os

r = subprocess.run(
    ['node', '3-memory-agent.mjs'],
    cwd=os.getcwd(), env=os.environ,
    capture_output=True, text=True, timeout=180,
)
print(r.stdout)
if r.stderr:
    print('STDERR:', r.stderr[:500])

## Summary

| Pattern | Key APIs | Use Case |
|---------|----------|----------|
| MCP Agent | `experimental_createMCPClient`, `generateText` | Natural-language graph queries |
| Custom Tools | `tool()`, `jsonSchema()`, `neo4j-driver` | Typed Cypher queries as agent tools |
| Memory Agent | `neo4j-driver`, before/after hooks | Cross-turn conversational memory |

All three patterns use the same `generateText` + `stopWhen: stepCountIs(N)` loop.  
Switch LLMs by changing `AI_PROVIDER` — see [`providers.mjs`](providers.mjs).